# Cache Transfer Workflow

This notebook demonstrates a common collaborative workflow using `CacheStack` and `transfer()`:

1. A **shared global cache** holds approved, vetted results (read-only for regular users).
2. Each user runs under a **local cache** stacked on top of the global one.
   - Cache hits in the global cache are served transparently.
   - New computations land only in the local cache.
3. After reviewing their local results, the user selects which ones to **promote** to the global cache via `transfer()`.

## 1. Setup

We create two caches:

- `_global_cache_rw` — a persistent SQL + PickleFile cache simulating a shared remote store (admin access only)
- `global_cache` — a `ReadOnlyCache` wrapper that regular users see (writes rejected)
- `local_cache` — a fast in-memory cache for the user's current session
- `user_cache` — a `CacheStack` combining local (priority) over global

In [ ]:
import tempfile
import os

from fleche import fleche, cache
from fleche.caches import Cache, CacheStack, ReadOnlyCache
from fleche.storage import Memory
from fleche.storage.pickle_file import PickleFile
from fleche.storage.sql import Sql

# Persistent storage for the global cache (simulates a shared remote store)
tmp_dir = tempfile.TemporaryDirectory()

_global_cache_rw = Cache(
    values=PickleFile.with_cloudpickle(tmp_dir.name),
    _calls=Sql(f"sqlite:///{os.path.join(tmp_dir.name, 'global.db')}"),
)

# Read-only view of the global cache — what regular users get
global_cache = ReadOnlyCache(_global_cache_rw)

# Fast in-memory cache for the current user session
local_cache = Cache(values=Memory({}), _calls=Memory({}))

# The user's active cache: local results take priority, global fills in the rest
user_cache = CacheStack((local_cache, global_cache))

print("Setup complete.")
print(f"  global cache (rw): {_global_cache_rw}")
print(f"  global cache (ro): {global_cache}")
print(f"  local  cache:      {local_cache}")
print(f"  user   cache:      {user_cache}")

## 2. Define Functions

Two `@fleche`-decorated functions simulate a heavy computation and a post-processing step.

In [ ]:
import time

@fleche
def simulate(param: float) -> dict:
    """Expensive simulation — takes time, results worth sharing."""
    print(f"  [simulate] Running simulation for param={param}...")
    time.sleep(0.05)  # pretend this is expensive
    return {"param": param, "result": param ** 2 + 1.0}


@fleche
def postprocess(data: dict) -> float:
    """Quick post-processing — user-specific, not worth sharing."""
    print(f"  [postprocess] Processing {data}...")
    return data["result"] * 2.0

## 3. Seed the Global Cache (Admin Step)

An admin pre-populates the global cache with approved baseline results.
Regular users never do this — they only read from `global_cache` (the `ReadOnlyCache`).

In [ ]:
print("Seeding global cache with approved results...")
with cache(_global_cache_rw):
    for p in [1.0, 2.0, 3.0]:
        simulate(p)

print(f"\nGlobal cache contents ({len(list(_global_cache_rw.table().index))} entries):")
_global_cache_rw.table()

## 4. User Session

The user runs under `user_cache = CacheStack((local_cache, global_cache))`.

- Params **already in the global cache** (1.0, 2.0, 3.0) → cache hits, no recomputation.
- **New params** (4.0, 5.0, 6.0) → computed and saved to `local_cache` only.
- Post-processing results also land in `local_cache`.

In [ ]:
print("User session starting under user_cache (CacheStack)...\n")

with cache(user_cache):
    print("--- Params already in global cache (should be cache hits) ---")
    for p in [1.0, 2.0, 3.0]:
        result = simulate(p)
        pp = postprocess(result)
        print(f"  simulate({p}) -> {result},  postprocess -> {pp}")

    print()
    print("--- New params (will be computed and saved locally) ---")
    for p in [4.0, 5.0, 6.0]:
        result = simulate(p)
        pp = postprocess(result)
        print(f"  simulate({p}) -> {result},  postprocess -> {pp}")

## 5. Inspect the Caches

After the session:
- The **global cache** is unchanged (still only has params 1–3).
- The **local cache** has all new results (simulate + postprocess for params 4–6, and postprocess for 1–3 which were hits from global).

In [ ]:
print("=== Global cache (unchanged) ===")
display(_global_cache_rw.table())

print("\n=== Local cache (user's new results) ===")
display(local_cache.table())

## 6. Filter Before Transfer

The user reviews their local cache and decides only the `simulate` results are worth promoting to the global cache.
Post-processing results are user-specific and stay local.

`filter()` returns a `FilteredCache` — a read-only view that `transfer()` will iterate over.

In [ ]:
# Keep only 'simulate' entries from the local cache
simulations_only = local_cache.filter(lambda c: c.name == "simulate")

print("Entries selected for promotion:")
display(simulations_only.table())

## 7. Transfer to the Global Cache

The user (or admin) calls `transfer()` to promote the selected results into the global cache.
After this, any user will get cache hits for `simulate(4.0)`, `simulate(5.0)`, and `simulate(6.0)` without recomputation.

In [ ]:
print("Transferring selected simulations to global cache...")
simulations_only.transfer(_global_cache_rw)

print("\n=== Global cache after transfer ===")
display(_global_cache_rw.table())

## 8. Verify: Second User Gets Cache Hits

A second user starts a fresh session with their own empty local cache.
They can now load `simulate(4.0)` – `simulate(6.0)` without any computation.

In [ ]:
# Second user has a fresh local cache
local_cache_2 = Cache(values=Memory({}), _calls=Memory({}))
user_cache_2 = CacheStack((local_cache_2, global_cache))

print("Second user session (all params 1–6 should now be cache hits)...\n")
with cache(user_cache_2):
    for p in [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]:
        result = simulate(p)
        print(f"  simulate({p}) -> {result}  (no recomputation)")

# Cleanup
tmp_dir.cleanup()
print("\nDone. Temporary directory cleaned up.")